<a href="https://colab.research.google.com/github/shikchit123/Data_science_5Months/blob/main/Month_01/Day_01_SQL_Joins_and_Grouping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Day 01: SQL JOINS AND AGGREGATION IN PYTHON

In [1]:
import pandas as pd
import sqlite3


**1. Create an in-memory SQLite Database connection**





In [2]:
conn = sqlite3.connect(":memory:")

# **Create Sample DataFrames (simulating two relational database tables)**

**Table A: Patients Information**

In [3]:
patients_data = {
    "patient_id": [101, 102, 103, 104, 105],
    "patient_name": ["samir", "Usha", "Hema", "Nischal", "Goma"],
    "age": [21, 20, 22, 19, 40],
    "city": ["Belaka", "Gorkha", "Birtamode", "Pakhribas", "Damak"],

}

**Table B: Clinical Drug Trial Results**

In [4]:
trials_data = {
    "trial_id": [1, 2, 3, 4, 5,6],
    "patient_id": [101, 101, 102, 103, 104, 105],
    "dosage_mg": [50, 100, 100, 50, 150, 50],
    "response": [1, 1, 0, 1, 1, 0], # 1 = Effective, 0 = Ineffective
}

patients_df = pd.DataFrame(patients_data)
trials_df = pd.DataFrame(trials_data)

# Store DataFrames as SQL Tables in SQLite Database

In [6]:
patients_df.to_sql("patients", conn, index=False, if_exists="replace")
trials_df.to_sql("trials", conn, index=False, if_exists="replace")

print("--- SQL Setup Complete: 'patients' and 'trials' tables created ---\n")

--- SQL Setup Complete: 'patients' and 'trials' tables created ---





---


 # SQL TASK 1: INNER JOIN
 # join patient demographic details with trial results

---



In [7]:
query_join = """
SELECT
    p.patient_id,
    p.patient_name,
    p.age,
    t.dosage_mg,
    t.response
FROM patients p
INNER JOIN trials t ON p.patient_id = t.patient_id;
"""

df_joined = pd.read_sql_query(query_join, conn)
print("1. combined patient Trial Data (INNER JOIN):")
print(df_joined)
print("\n" + "=" * 50 + "\n")


1. combined patient Trial Data (INNER JOIN):
   patient_id patient_name  age  dosage_mg  response
0         101        samir   21         50         1
1         101        samir   21        100         1
2         102         Usha   20        100         0
3         103         Hema   22         50         1
4         104      Nischal   19        150         1
5         105         Goma   40         50         0






---
# SQL TASK 2: GROUP BY AND AGGREGATION
# calculate average success rate and total trials per dosage


---



In [13]:
query_aggregation = """
SELECT
    dosage_mg,
    COUNT(trial_id) AS total_trials,
    AVG(response) AS success_rate
FROM trials
GROUP BY dosage_mg
ORDER BY success_rate DESC;
"""

df_aggregated = pd.read_sql_query(query_aggregation, conn)
print("2. success Rate and Total Tests grouped by Dosage (Group BY):")
print(df_aggregated)
print("\n" + "=" * 50 + "\n")

2. success Rate and Total Tests grouped by Dosage (Group BY):
   dosage_mg  total_trials  success_rate
0        150             1      1.000000
1         50             3      0.666667
2        100             2      0.500000






---
# SQL TASK 3: JOIN + WHERE + GROUP BY
# Analyze patient response rates by city for dosages >= 50mg


---



In [14]:
query_complex = """
SELECT
    p.city,
    COUNT(t.trial_id) AS total_patients_tested,
    AVG(t.response) AS city_success_rate
FROM patients p
JOIN trials t ON p.patient_id = t.patient_id
WHERE t.dosage_mg >= 50
GROUP BY p.city
HAVING total_patients_tested >= 1;
"""

df_complex = pd.read_sql_query(query_complex, conn)
print("3. city-level performance Analysis (JOIN + WHERE +GROUP BY + HAVING):")
print(df_complex)


3. city-level performance Analysis (JOIN + WHERE +GROUP BY + HAVING):
        city  total_patients_tested  city_success_rate
0     Belaka                      2                1.0
1  Birtamode                      1                1.0
2      Damak                      1                0.0
3     Gorkha                      1                0.0
4  Pakhribas                      1                1.0
